# Exploring Registrar

This notebook demonstrates how to use the `Registrar` class to load a manifest and run dataflows.
The example manifest (`examples/example/manifest.yaml`) models a cat-happiness device with requirements and solutions.

In [ ]:
import ibis

from iacs.registrar import Registrar

ibis.options.interactive = True

## Load the Example Manifest

`Registrar.from_manifest` reads the YAML files and builds the registry. Then call `load_dataflow` with a module name string to attach Hamilton dataflow modules from `iacs.dataflows`.

In [ ]:
registrar = Registrar.from_manifest("../examples/example")

# registrar.load_dataflow("audit.traceability")
# registrar.load_dataflow("audit.todo")
# registrar.load_dataflow("audit.requirement_coverage")
registrar.load_dataflow("export_manifest")

## Inspect the Registry

The registry holds the loaded data in component-centered format — one table per component type.

In [ ]:
# Component types present in the manifest
registrar.registry.component_types

In [ ]:
# View a specific component table
registrar.registry.view("description")

In [ ]:
registrar.registry.view("requirement")

## Explore the Dataflows

The dataflows attached to the registrar are Hamilton DAGs. You can visualize the full graph of available computations.

In [ ]:
# List all computable outputs (non-input nodes in the DAG)
registrar.outputs

In [ ]:
# Visualize the full dataflow DAG
registrar._driver.display_all_functions(orient="TB")

## Execute Dataflows

Call `registrar.execute` with a list of output names to compute results.

In [ ]:
# Run the traceability audit — entities not linked to any requirement show up here
results = registrar.execute(["traceability"])
results["traceability"]

In [ ]:
# Run the todo audit — outstanding todos in the manifest
results = registrar.execute(["todo"])
results["todo"]

In [ ]:
# Run multiple outputs at once
results = registrar.execute(["all_entities", "req_entities", "solution_entities", "orphan_entities"])
for name, table in results.items():
    print(f"--- {name} ---")
    print(table.execute())
    print()

## Visualize Execution Paths

You can also visualize the subgraph that Hamilton will execute for a specific output.

In [ ]:
registrar._driver.visualize_execution(["traceability"])